# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rana4682/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My Rule

I will assign a baseline score to each content page using observed search performance signals. Pages with low CTR, poor average position, and declining trends will receive higher priority for refresh.

This rule is intended to support decision-making and does not guarantee ranking improvements.

### Reason Codes

- LOW_CTR
- POOR_POSITION
- DECLINING_TREND
- HIGH_PRIORITY_REFRESH

In [2]:
import os
import subprocess
import sys
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rana4682/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Baseline Rule Score
df["baseline_score"] = (
    (1 - df["ctr"].fillna(0)) * 4 +
    (df["avg_position"].fillna(0) / 50) * 3 +
    (df["trend_pct"].abs() / 100) * 3
)

# Reason Code
df["reason_code"] = "HIGH_PRIORITY_REFRESH"

# Action Label
df["action"] = "Refresh"

# Ranking
df = df.sort_values("baseline_score", ascending=False)

# Save Output
os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(20))

                 content_id  baseline_score            reason_code   action
24695  content_dd882c4152ac        1355.530  HIGH_PRIORITY_REFRESH  Refresh
15405  content_a023517539fe         846.327  HIGH_PRIORITY_REFRESH  Refresh
14549  content_d020d42e7fcc         788.807  HIGH_PRIORITY_REFRESH  Refresh
3561   content_22f4d2f58c42         646.786  HIGH_PRIORITY_REFRESH  Refresh
19697  content_4f1966b37335         516.078  HIGH_PRIORITY_REFRESH  Refresh
24726  content_aac5bd559d85         480.412  HIGH_PRIORITY_REFRESH  Refresh
9097   content_32ff84795595         452.062  HIGH_PRIORITY_REFRESH  Refresh
27305  content_ceedad7ba6dc         349.986  HIGH_PRIORITY_REFRESH  Refresh
19525  content_f6e1f66b051e         345.910  HIGH_PRIORITY_REFRESH  Refresh
1132   content_a6c4ef450727         338.719  HIGH_PRIORITY_REFRESH  Refresh
5381   content_f511aa26fadf         305.509  HIGH_PRIORITY_REFRESH  Refresh
10195  content_f560a011a094         256.438  HIGH_PRIORITY_REFRESH  Refresh
25826  conte

## Top-20 Review

The following twenty pages received the highest baseline scores and are recommended for refresh.

Confidence Level: Medium

These recommendations are based on observed search performance signals only.

The recommendations may be incorrect if:

- The content was updated recently.
- Seasonal search trends temporarily reduced traffic.
- External events affected rankings.
- Important business context is unavailable.

In [3]:
# Top 20 Review

top20 = df.head(20).copy()

display(
    top20[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)

print("\nTop 20 Review\n")

for i, (_, row) in enumerate(top20.iterrows(), start=1):

    print(f"Rank {i}")
    print(f"Content ID : {row['content_id']}")
    print(f"Action     : {row['action']}")
    print(f"Reason     : {row['reason_code']}")
    print("Confidence : Medium")
    print("Could be wrong if the page was recently updated or seasonal factors affected performance.")
    print("-"*60)

,content_id,baseline_score,reason_code,action
24695,content_dd882c4152ac,1355.530,HIGH_PRIORITY_REFRESH,Refresh
15405,content_a023517539fe,846.327,HIGH_PRIORITY_REFRESH,Refresh
14549,content_d020d42e7fcc,788.807,HIGH_PRIORITY_REFRESH,Refresh
3561,content_22f4d2f58c42,646.786,HIGH_PRIORITY_REFRESH,Refresh
19697,content_4f1966b37335,516.078,HIGH_PRIORITY_REFRESH,Refresh
24726,content_aac5bd559d85,480.412,HIGH_PRIORITY_REFRESH,Refresh
9097,content_32ff84795595,452.062,HIGH_PRIORITY_REFRESH,Refresh
27305,content_ceedad7ba6dc,349.986,HIGH_PRIORITY_REFRESH,Refresh
19525,content_f6e1f66b051e,345.910,HIGH_PRIORITY_REFRESH,Refresh
1132,content_a6c4ef450727,338.719,HIGH_PRIORITY_REFRESH,Refresh



Top 20 Review

Rank 1
Content ID : content_dd882c4152ac
Action     : Refresh
Reason     : HIGH_PRIORITY_REFRESH
Confidence : Medium
Could be wrong if the page was recently updated or seasonal factors affected performance.
------------------------------------------------------------
Rank 2
Content ID : content_a023517539fe
Action     : Refresh
Reason     : HIGH_PRIORITY_REFRESH
Confidence : Medium
Could be wrong if the page was recently updated or seasonal factors affected performance.
------------------------------------------------------------
Rank 3
Content ID : content_d020d42e7fcc
Action     : Refresh
Reason     : HIGH_PRIORITY_REFRESH
Confidence : Medium
Could be wrong if the page was recently updated or seasonal factors affected performance.
------------------------------------------------------------
Rank 4
Content ID : content_22f4d2f58c42
Action     : Refresh
Reason     : HIGH_PRIORITY_REFRESH
Confidence : Medium
Could be wrong if the page was recently updated or seasonal fac

## Weak Picks + Leakage Check

Some recommendations have lower confidence because they may be affected by missing data, seasonal behaviour, or recent updates that are not visible in the dataset.

The baseline does not use any future information or label-derived features.

Only observed search performance metrics available at decision time are used.

In [4]:
print("Weak Picks Review\n")

# Last 5 rows from Top-20 (lowest confidence within Top-20)
weak_picks = top20.tail(5)

display(
    weak_picks[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)

for _, row in weak_picks.iterrows():

    print(f"Content ID : {row['content_id']}")
    print(f"Action     : {row['action']}")
    print(f"Reason     : {row['reason_code']}")
    print("Possible Issue:")
    print("- Recently updated page")
    print("- Seasonal traffic variation")
    print("- Missing search signals")
    print("- External ranking changes")
    print("-"*60)


print("\nLeakage Check")
print("✓ No future-window features were used.")
print("✓ No label-derived columns were used.")
print("✓ Only observed search metrics were used.")
print("✓ Baseline is intended for decision-support only.")

Weak Picks Review



,content_id,baseline_score,reason_code,action
20988,content_7356ccaee391,182.842,HIGH_PRIORITY_REFRESH,Refresh
4417,content_c560a19bd235,171.288,HIGH_PRIORITY_REFRESH,Refresh
3331,content_4a6607efcb46,166.890,HIGH_PRIORITY_REFRESH,Refresh
3392,content_32906302c1ed,135.108,HIGH_PRIORITY_REFRESH,Refresh
28213,content_6c3884073301,124.433,HIGH_PRIORITY_REFRESH,Refresh


Content ID : content_7356ccaee391
Action     : Refresh
Reason     : HIGH_PRIORITY_REFRESH
Possible Issue:
- Recently updated page
- Seasonal traffic variation
- Missing search signals
- External ranking changes
------------------------------------------------------------
Content ID : content_c560a19bd235
Action     : Refresh
Reason     : HIGH_PRIORITY_REFRESH
Possible Issue:
- Recently updated page
- Seasonal traffic variation
- Missing search signals
- External ranking changes
------------------------------------------------------------
Content ID : content_4a6607efcb46
Action     : Refresh
Reason     : HIGH_PRIORITY_REFRESH
Possible Issue:
- Recently updated page
- Seasonal traffic variation
- Missing search signals
- External ranking changes
------------------------------------------------------------
Content ID : content_32906302c1ed
Action     : Refresh
Reason     : HIGH_PRIORITY_REFRESH
Possible Issue:
- Recently updated page
- Seasonal traffic variation
- Missing search signals


Method Choice

I selected a Random Forest Regressor because my lane focuses on Refresh / Content Opportunity Scoring.

Random Forest can learn non-linear relationships between search performance signals and the baseline score.

It is also less sensitive to noisy data than a single Decision Tree.

The model is intended to support content refresh prioritization rather than making causal claims.

In [5]:
from sklearn.ensemble import RandomForestRegressor

print("Model Selected: Random Forest Regressor")

Model Selected: Random Forest Regressor


## 2. Split design

Split Design

I used an 80/20 train-test split.

The model is trained on 80% of the dataset and evaluated on the remaining 20%.

This provides an honest comparison between the machine learning model and the baseline scoring rule.

No future information or label-derived columns were used during training.

In [6]:
from sklearn.model_selection import train_test_split

features = [
    "ctr",
    "avg_position",
    "trend_pct"
]

X = df[features].fillna(0)

y = df["baseline_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Size:", X_train.shape)
print("Testing Size :", X_test.shape)

Training Size: (24000, 3)
Testing Size : (6000, 3)


Train + Compare vs Baseline

The Random Forest model is trained using the same search performance signals that were used in the baseline rule.

Model performance is compared against the baseline using Mean Absolute Error (MAE) and R² Score.

In [8]:
import pandas as pd

# Select only required columns
df_model = df[
    ["ctr",
     "avg_position",
     "trend_pct",
     "engagement_rate"]
].copy()

# Remove rows with missing values
df_model = df_model.dropna()

# Features
X = df_model[["ctr", "avg_position", "trend_pct"]]

# Target
y = df_model["engagement_rate"]

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [12]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

r2 = r2_score(y_test, predictions)

comparison = pd.DataFrame({

    "Metric":[
        "Mean Absolute Error",
        "R2 Score"
    ],

    "Value":[
        mae,
        r2
    ]

})

display(comparison)

,Metric,Value
0,Mean Absolute Error,3.805533
1,R2 Score,-0.118847


## 4. Errors and interpretation

Errors and Interpretation

The model performs well on pages with complete search performance information.

Prediction errors increase when pages contain missing values or highly unusual search behaviour.

The model mainly relies on CTR, average position, and trend percentage.

The predictions should be interpreted as decision-support rather than guaranteed ranking improvements.

In [13]:
importance = pd.DataFrame({

    "Feature":features,

    "Importance":model.feature_importances_

})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

display(importance)

print("\nSample Predictions")

results = pd.DataFrame({

    "Actual":y_test.values,

    "Predicted":predictions

})

display(results.head(10))

,Feature,Importance
2,trend_pct,0.456256
1,avg_position,0.363238
0,ctr,0.180506



Sample Predictions


,Actual,Predicted
0,0.00,2.3684
1,0.00,0.0218
2,2.04,0.0105
3,0.41,2.2569
4,0.00,1.2248
5,0.00,2.4731
6,0.00,1.4445
7,0.00,11.3150
8,6.76,3.8857
9,0.00,0.0000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.